In [1]:
!pip install sqlalchemy requests kafka-python

In [3]:
from sqlalchemy import create_engine, Column, Integer, String, Float, Boolean, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime

# 1. Połączenie z bazą (stworzy się plik 'crypto_alerts.db' w folderze)
engine = create_engine('sqlite:///crypto_alerts.db', echo=False)
Base = declarative_base()

# 2. Definicja tabeli z alertami
class AlertRule(Base):
    __tablename__ = 'alert_rules'
    id = Column(Integer, primary_key=True)
    user_id = Column(String)           
    symbol = Column(String)            # np. "BTCUSDT"
    condition_type = Column(String)    # "ABOVE" lub "BELOW"
    threshold = Column(Float)          # próg cenowy
    is_active = Column(Boolean, default=True) # czy alert wciąż czeka na realizację
    created_at = Column(DateTime, default=datetime.utcnow)

# 3. Utworzenie tabeli i sesji
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

print("Baza danych gotowa! Tabele zostały utworzone.")

Baza danych gotowa! Tabele zostały utworzone.


In [4]:
import requests

#SKOPIOWANY LINK
WEBHOOK_URL = "https://discord.com/api/webhooks/1512932247051702366/4fuUNnlzyJlWE8Xf-mLFVtmZhkoz9iul9lEM1ry9aC_Tjim5Y5ZJCZ4SNtforfVoEswC"

def send_discord_alert(symbol, price, condition, threshold, user_id):
    """Wysyła sformatowaną wiadomość na kanał Discord."""
    kierunek = "wzrosła POWYŻEJ" if condition == "ABOVE" else "spadła PONIŻEJ"
    
    wiadomosc = (
        f"🚨 **ALERT SYSTEMOWY** 🚨\n"
        f"Użytkownik: `{user_id}`\n"
        f"Kryptowaluta: **{symbol}**\n"
        f"Zdarzenie: Cena {kierunek} poziomu {threshold}$!\n"
        f"Aktualna cena giełdowa: **{price}$** 📈"
    )
    
    data = {"content": wiadomosc, "username": "Krypto Bot Alertowy"}
    
    response = requests.post(WEBHOOK_URL, json=data)
    if response.status_code == 204:
        print(f"✅ [Discord] Wysłano powiadomienie dla {symbol}!")
    else:
        print(f"❌ [Discord] Błąd wysyłania: {response.status_code}")

In [5]:
# Czyszczenie bazy przed testem
session.query(AlertRule).delete()

# Dodajemy alert - ustawiamy próg na np.104500$
alert1 = AlertRule(user_id="Marta", symbol="BTCUSDT", condition_type="ABOVE", threshold=63542.0)

session.add_all([alert1])
session.commit()

print("Testowy alert na realną cenę został zapisany w bazie!")

Testowy alert na realną cenę został zapisany w bazie!


In [6]:
import json
from kafka import KafkaConsumer

def check_prices(symbol, current_price):
    """Pobiera aktywne alerty z bazy i sprawdza warunki."""
    active_rules = session.query(AlertRule).filter(
        AlertRule.symbol == symbol,
        AlertRule.is_active == True
    ).all()
    
    for rule in active_rules:
        trigger = False
        
        # Czy cena z Kafki przekroczyła próg
        if rule.condition_type == "ABOVE" and current_price > rule.threshold:
            trigger = True
        elif rule.condition_type == "BELOW" and current_price < rule.threshold:
            trigger = True
            
        if trigger:
            # Jeśli warunek jest spełniony, bot wysyła wiadomość na Discorda
            send_discord_alert(rule.symbol, current_price, rule.condition_type, rule.threshold, rule.user_id)
            # Wyłączamy alert w bazie, żeby nie wysyłał spamu co sekundę
            rule.is_active = False
            session.commit()

# Łączymy się z serwerem Kafki uruchomionym w Terminalu
print("Inicjalizowanie połączenia z Kafką...")
consumer = KafkaConsumer(
    "btc.prices", 
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    auto_offset_reset="latest",              # Odbieramy tylko najnowsze ceny na żywo
    group_id="marta-alert-consumer"          # Twoje unikalne ID jako konsumenta w grupie
)

print("Rozpoczynam nasłuchiwanie cen Bitcoina z Kafki w czasie rzeczywistym... \n")

# Pętla działa bez przerwy i łapie każdy ruch ceny wysłany przez grupę
for msg in consumer:
    data = msg.value
    symbol = data.get("symbol")   # To będzie "BTCUSDT"
    price = data.get("price")     # aktualna cena giełdowa
    
    print(f"Otrzymano z Kafki | {symbol}: {price:.2f}$")
    
    if symbol and price:
        check_prices(symbol, price)

Inicjalizowanie połączenia z Kafką...
Rozpoczynam nasłuchiwanie cen Bitcoina z Kafki w czasie rzeczywistym... 

Otrzymano z Kafki | BTCUSDT: 63564.55$
✅ [Discord] Wysłano powiadomienie dla BTCUSDT!
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BTCUSDT: 63564.54$
Otrzymano z Kafki | BT

KeyboardInterrupt: 